# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


In [100]:
# imports

import os
import io
import sys
import gradio as gr
import subprocess
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display


In [101]:
load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not google_api_key:
    print("Error: GOOGLE_API_KEY is not set in the environment variables.")
    sys.exit(1)

if not groq_api_key:
    print("Error: GROQ_API_KEY is not set in the environment variables.")
    sys.exit(1)

if not openrouter_api_key:
    print("Error: OPENROUTER_API_KEY is not set in the environment variables.")
    sys.exit(1)

In [130]:
# Connect to client libraries

openai = OpenAI()

groq_url = "https://api.groq.com/openai/v1"
openrouter_url = "https://openrouter.ai/api/v1"

groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


In [132]:
models = ["llama-3.3-70b-versatile", "qwen/qwen3-32b", "qwen/qwen3-coder-30b-a3b-instruct"]

clients = { "llama-3.3-70b-versatile": groq, "qwen/qwen3-32b": groq, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}



In [104]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-windows-gnu'},
 'package_managers': ['winget'],
 'cpu': {'brand': '12th Gen Intel(R) Core(TM) i5-1235U',
  'cores_logical': 12,
  'cores_physical': 10,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (MinGW-W64 x86_64-ucrt-posix-seh, built by Brecht Sanders, r2) 14.2.0',
   'g++': 'g++.EXE (MinGW-W64 x86_64-ucrt-posix-seh, built by Brecht Sanders, r2) 14.2.0',
   'clang': '(built by Brecht Sanders, r2) clang version 19.1.1',
   'msvc_cl': ''},
  'build_tools': {'cmake': 'cmake version 3.30.4',
   'ninja': '1.12.1',
   'make': ''},
  'linkers': {'ld_lld': '(built by Brecht Sanders, r2) LLD 19.1.1 (compatible with GNU linkers)'}}}

In [105]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openrouter.chat.completions.create(model="qwen/qwen3-coder-30b-a3b-instruct", messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

Based on your system information, you **already have a C++ compiler installed**. Specifically, you have `g++` (GNU C++ compiler) available, which is part of the MinGW-w64 toolchain.

## You don't need to install anything else.

## Here's the exact Python code you should use:

```python
import subprocess

compile_command = ["g++", "-O3", "-std=c++17", "main.cpp", "-o", "main.exe"]
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = ["./main.exe"]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```

## Explanation of flags:
- `-O3`: Enables maximum optimization for fastest runtime performance
- `-std=c++17`: Uses C++17 standard (modern and widely supported)
- `-o main.exe`: Specifies output executable name (Windows requires .exe extension)

## Notes:
1. Your system is Windows 11 with AMD64 architecture
2. The `g++` command is already available in your PATH
3. On Windows, the executable will be named `main.exe` (not `main`)
4. The `-O3` flag gives you the fastest possible runtime performance
5. Use `./main.exe` to run it (the `./` is needed on Windows when using subprocess)

## To run this in a single command line for testing:
```bash
g++ -O3 -std=c++17 main.cpp -o main.exe && main.exe
```

You're all set to compile and run C++ code with optimal performance!

In [121]:
run_command = ['./main.exe']

compile_command = [
    'clang++',
    '-O3',
    '-march=native',
    '-flto',
    '-o', 'main.exe',
    'main.cpp'
]

In [ ]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [108]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [109]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [110]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [125]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [112]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [113]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [114]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [133]:
import gradio as gr

# CSS for pixel-perfect alignment adjustments
custom_css = """
.container { padding: 20px; }
.button-row { display: flex; align-items: flex-end; }
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as ui:
    gr.Markdown("# Python to C++ Code Converter with LLMs")
    
    # 1. Main Work Area (Equal 1:1 split)
    with gr.Row():
        with gr.Column(scale=1):
            python_input = gr.Code(
                label="Source: Python", 
                language="python", 
                lines=25, 
                value=pi
            )
        with gr.Column(scale=1):
            cpp_output = gr.Code(
                label="Output: C++", 
                language="cpp", 
                lines=25
            )

    # 2. Control Row (Aligned and proportional)
    with gr.Row(variant="panel"):
        # Scale 4:1 gives the dropdown dominance, button remains tight
        with gr.Column(scale=4):
            model_select = gr.Dropdown(
                choices=models, 
                label="Intelligence Engine", 
                value=models[0],
                container=True
            )
        with gr.Column(scale=1):
            # A blank component to push the button right if desired, 
            # or just let it hug the dropdown
            convert_btn = gr.Button("🚀 Convert Code", variant="primary", size="lg")

    # 3. Connection
    convert_btn.click(
        fn=port, 
        inputs=[model_select, python_input], 
        outputs=[cpp_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


In [129]:
compile_and_run()

An error occurred:
clang++: error: 'x86_64-w64-windows-gnu': unable to pass LLVM bit-code files to linker

